In [1]:
from pathlib import Path

BASE_DIR = Path("data_lake")
TARGET_DIRS = [
    BASE_DIR / "bronze",
    BASE_DIR / "silver" / "preprocessed_sales",
]

for directory in TARGET_DIRS:
    directory.mkdir(parents=True, exist_ok=True)

print("Folder structure created successfully:\n")

def display_tree(root_path: Path, prefix: str = ""):
    print(f"{root_path.name}/")
    items = sorted([p for p in root_path.iterdir() if p.is_dir()])
    for index, path in enumerate(items):
        is_last = index == len(items) - 1
        connector = "└── " if is_last else "├── "
        extension = "    " if is_last else "│   "
        print(f"{prefix}{connector}", end="")
        display_tree(path, prefix + extension)


display_tree(BASE_DIR)

Folder structure created successfully:

data_lake/
├── bronze/
└── silver/
    └── preprocessed_sales/


In [2]:
from google.colab import files
import shutil
import os

print("Please upload your 'sales.csv' file below:")
uploaded = files.upload()

for filename in uploaded.keys():
    target_path = os.path.join('data_lake/bronze', filename)
    shutil.move(filename, target_path)
    print(f"\nSuccessfully uploaded and moved '{filename}' to '{target_path}'")

Please upload your 'sales.csv' file below:


Saving sales.csv to sales.csv

Successfully uploaded and moved 'sales.csv' to 'data_lake/bronze/sales.csv'


In [3]:
!pip install pyspark

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("DataLakeBronzeExploration") \
    .getOrCreate()

bronze_path = "data_lake/bronze/sales.csv"
df = spark.read.csv(bronze_path, header=True, inferSchema=True)

print("=== DataFrame Schema ===")
df.printSchema()

print(f"Total Record Count: {df.count()}")

print("\n=== Sample Records ===")
df.show(5, truncate=False)

=== DataFrame Schema ===
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

Total Record Count: 1000

=== Sample Records ===
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-----------

In [5]:
# 1. Identify Duplicates
duplicate_count = df.count() - df.dropDuplicates().count()
print(f"Number of exact duplicate rows: {duplicate_count}")

Number of exact duplicate rows: 10


In [6]:
# 2. Identify Null Values per Column
print("=== Null Values Count per Column ===")
from pyspark.sql import functions as F

null_counts = [
    F.sum(
        F.when(
            F.col(c).isNull() | (F.col(c).cast("string") == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df.columns
]

df.select(null_counts).show()

=== Null Values Count per Column ===
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [7]:
# 3. Analyze Text Format Inconsistencies & Numerical Anomalies
from pyspark.sql import functions as F

print("=== Unique Category Values ===")
df.groupBy("category").count().show(truncate=False)

print("=== Numerical Anomalies ===")
num_cols = [c for c, t in df.dtypes if t in ["int", "bigint", "float", "double"]]

for c in num_cols:
    print(f"\nStats for {c}:")
    df.select(
        F.min(c).alias("min"),
        F.max(c).alias("max"),
        F.sum(F.when(F.col(c) < 0, 1).otherwise(0)).alias("negative_count")
    ).show()

=== Unique Category Values ===
+---------------+-----+
|category       |count|
+---------------+-----+
|SPORTS         |1    |
|beauty         |2    |
|Sports         |136  |
|Grocery        |136  |
|Electronics    |103  |
|Clothing       |108  |
|Books          |128  |
|ELECTRONICS    |1    |
|Home Appliances|115  |
|Furniture      |132  |
|sports         |1    |
|Beauty         |136  |
|CLOTHING       |1    |
+---------------+-----+

=== Numerical Anomalies ===

Stats for order_id:
+------+------+--------------+
|   min|   max|negative_count|
+------+------+--------------+
|100001|101000|             0|
+------+------+--------------+


Stats for quantity:
+---+---+--------------+
|min|max|negative_count|
+---+---+--------------+
| -2| 10|             6|
+---+---+--------------+


Stats for unit_price:
+----+-----+--------------+
| min|  max|negative_count|
+----+-----+--------------+
|-500|81010|             4|
+----+-----+--------------+


Stats for discount_percent:
+---+---+------

In [8]:
date_cols = [c for c in df.columns if "date" in c.lower()]

for c in date_cols:
    print(f"\n=== Checking {c} (Correct format: yyyy-MM-dd) ===")
    parsed = F.try_to_timestamp(F.col(c), F.lit("yyyy-MM-dd"))

    df.select(
        F.sum(F.when(parsed.isNull() & F.col(c).isNotNull(), 1).otherwise(0)).alias("invalid_dates"),
        F.min(parsed).alias("min_date"),
        F.max(parsed).alias("max_date")
    ).show()


=== Checking order_date (Correct format: yyyy-MM-dd) ===
+-------------+-------------------+-------------------+
|invalid_dates|           min_date|           max_date|
+-------------+-------------------+-------------------+
|            6|2025-01-01 00:00:00|2026-09-30 00:00:00|
+-------------+-------------------+-------------------+



In [9]:
print("=== Removing Duplicate Rows ===")
df = df.dropDuplicates()

=== Removing Duplicate Rows ===


In [10]:
print("=== Handling Missing Values ===")

text_cols = [c for c, t in df.dtypes if t == "string"]
num_cols = [c for c, t in df.dtypes if t in ["int", "bigint", "float", "double"]]

# Text columns
df = df.fillna("Unknown", subset=text_cols)

# Numerical columns
for c in num_cols:
    median = df.approxQuantile(c, [0.5], 0.01)[0]
    df = df.fillna({c: median})

=== Handling Missing Values ===


In [11]:
print("=== Cleaning Text Columns ===")

for c in text_cols:
    df = df.withColumn(c, F.trim(F.col(c)))
if "category" in df.columns:
    df = df.withColumn("category", F.initcap(F.col("category")))

=== Cleaning Text Columns ===


In [12]:
print("=== Validating Numerical Values ===")
from pyspark.sql import functions as F

df = df.withColumn(
    "quantity",
    F.when(F.col("quantity") > 0, F.col("quantity"))
)

df = df.withColumn(
    "unit_price",
    F.when(F.col("unit_price") >= 0, F.col("unit_price"))
)

df = df.withColumn(
    "discount_percent",
    F.when(
        F.col("discount_percent").between(0, 100),
        F.col("discount_percent")
    )
)

=== Validating Numerical Values ===


In [13]:
print("=== Cleaning Date Columns ===")

safe_date = F.coalesce(
    F.try_to_timestamp(F.trim(F.col("order_date")), F.lit("yyyy-MM-dd")),
    F.try_to_timestamp(F.trim(F.col("order_date")), F.lit("yyyy-dd-MM")),
    F.try_to_timestamp(F.trim(F.col("order_date")), F.lit("dd/MM/yyyy")),
    F.try_to_timestamp(F.trim(F.col("order_date")), F.lit("yyyy/MM/dd")),
    F.try_to_timestamp(F.trim(F.col("order_date")), F.lit("MM-dd-yyyy"))
)

df = df.withColumn("order_date", safe_date.cast("date"))

=== Cleaning Date Columns ===


In [14]:
print("=== Post-Validation Diagnostic Check ===")
from pyspark.sql import functions as F

# Let's count how many null values exist in each column now
null_diagnostics = [
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]

print("Current Null counts across columns:")
df.select(null_diagnostics).show()

print("Summary of numerical ranges:")
for col_name in ["quantity", "unit_price", "discount_percent"]:
    df.select(
        F.min(col_name).alias(f"min_{col_name}"),
        F.max(col_name).alias(f"max_{col_name}")
    ).show()

=== Post-Validation Diagnostic Check ===
Current Null counts across columns:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|            0|      0|       0|      12|         4|               8|             0|   0|    0|         3|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+

Summary of numerical ranges:
+------------+------------+
|min_quantity|max_quantity|
+------------+------------+
|           1|          10|
+------------+------------+

+--------------+--------------+
|min_unit_price

In [15]:
print("=== Handling Numerical Nulls ===")

num_cols = ["quantity", "unit_price", "discount_percent"]

for c in num_cols:
    median = df.approxQuantile(c, [0.5], 0.01)[0]
    df = df.fillna({c: median})

=== Handling Numerical Nulls ===


In [16]:
print("=== Date Nulls Preserved ===")

df.filter(F.col("order_date").isNull()).show()

=== Date Nulls Preserved ===
+--------+-----------+-------------+-------------+---------+--------+----------+----------------+----------------+---------+-------------+----------+------------+
|order_id|customer_id|customer_name|      product| category|quantity|unit_price|discount_percent|  payment_method|     city|        state|order_date|order_status|
+--------+-----------+-------------+-------------+---------+--------+----------+----------------+----------------+---------+-------------+----------+------------+
|  100097|      C0244|   Shreya Rao|         Sofa|Furniture|       4|     47430|              10|     Net Banking|Hyderabad|    Telangana|      NULL|    Returned|
|  100448|      C0270|  Pranav Iyer|Bedside Table|Furniture|       1|      4290|              35|             UPI|  Lucknow|Uttar Pradesh|      NULL|   Delivered|
|  100634|      C0262|  Tanvi Mehta|    Atta 10kg|  Grocery|      10|       560|              15|Cash on Delivery|Hyderabad|    Telangana|      NULL|  Proce

In [17]:
import shutil
import os
import glob
from pyspark.sql import functions as F

# Save the cleaned active DataFrame 'df'
silver_path = "data_lake/silver/preprocessed_sales"
temp_csv_path = "data_lake/silver/preprocessed_sales_temp"
final_file_name = "sales_silver.csv"
final_file_path = os.path.join(silver_path, final_file_name)

print("=== Saving Cleaned Dataset as CSV directly to preprocessed_sales ===")

# 1. Write the DataFrame as a single CSV file into a temporary directory
df.coalesce(1).write.mode("overwrite").option("header", "true").csv(temp_csv_path)

# 2. If the final directory already exists, clear it to avoid nested structures
if os.path.exists(silver_path):
    shutil.rmtree(silver_path)

# 3. Rename the temporary directory to the final preprocessed_sales folder
os.rename(temp_csv_path, silver_path)

# 4. Find the part CSV file in the directory and rename it to sales_silver.csv
part_files = glob.glob(os.path.join(silver_path, "part-*.csv"))
if part_files:
    os.rename(part_files[0], final_file_path)
    print(f"Successfully saved and renamed CSV directly to: {final_file_path}")

# 5. Clean up Spark metadata files and hidden CRC files
for item in os.listdir(silver_path):
    item_path = os.path.join(silver_path, item)
    if item != final_file_name:
        if os.path.isdir(item_path):
            shutil.rmtree(item_path)
        else:
            os.remove(item_path)

print("\n=== Contents of the Silver Zone Directory ===")
for root, dirs, files in os.walk('data_lake/silver'):
    level = root.replace('data_lake/silver', '').count(os.sep)
    indent = ' ' * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        if not f.startswith('.'):
            print(f"{indent}    {f}")

=== Saving Cleaned Dataset as CSV directly to preprocessed_sales ===
Successfully saved and renamed CSV directly to: data_lake/silver/preprocessed_sales/sales_silver.csv

=== Contents of the Silver Zone Directory ===
silver/
    preprocessed_sales/
        sales_silver.csv


In [18]:
print("=== Reading Preprocessed Silver Data (CSV) ===")
silver_path = "data_lake/silver/preprocessed_sales"
silver_df = spark.read.csv(silver_path, header=True, inferSchema=True)

print(f"Silver Dataset Record Count: {silver_df.count()}")
print("\n=== Schema ===")
silver_df.printSchema()

print("\n=== Sample Records ===")
silver_df.show(5, truncate=False)

=== Reading Preprocessed Silver Data (CSV) ===
Silver Dataset Record Count: 990

=== Schema ===
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)


=== Sample Records ===
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category |quantity|unit_price|discount_percent|payment_method  |city      |state      |ord